# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# List available Record Sets and their fields/columns, referencing by @id
record_sets = dataset.record_sets()

print('Record sets in this dataset:')
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id} | name: {rs.name}")
    if hasattr(rs, 'fields') and rs.fields:
        print('   Fields:')
        for field in rs.fields:
            print(f"      - Field @id: {field.id} | name: {field.name} | data_type: {field.data_type}")
    elif hasattr(rs, 'columns') and rs.columns:
        print('   Columns:')
        for col in rs.columns:
            print(f"      - Column @id: {col.id} | name: {col.name} | data_type: {col.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Use the @id of the primary Record Set -- edit if your dataset structure changes.
# Here, we identify the main table by listing available record set IDs.
record_set_ids = [rs.id for rs in dataset.record_sets()]

# We'll load all of them for demonstration
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

print(f"Available DataFrames: {list(dataframes.keys())}")

# Pick a main record set for demonstration
main_record_set_id = list(dataframes.keys())[0] if dataframes else None
if main_record_set_id is not None:
    print(f"Columns in {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No records found in any record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note:** Change the field IDs if your main record set or numeric fields differ.

In [ ]:
# Identify available numeric fields by inspecting the DataFrame
import numpy as np
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_fields:
        print("No numeric fields detected in main DataFrame.")
    else:
        print(f"Numeric fields: {numeric_fields}")
        # Select a numeric field for demonstration
        numeric_field = numeric_fields[0]

        # Example: Filter values above a threshold (default: mean)
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a likely categorical column
        candidate_group_fields = [c for c in df.columns if df[c].dtype == 'object' and c != numeric_field]
        group_field = candidate_group_fields[0] if candidate_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping was possible, draw mean per group
    if 'grouped_df' in locals() and group_field is not None:
        grouped_df = grouped_df.reset_index()
        plt.figure(figsize=(12,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} per {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using `mlcroissant`, we loaded metadata and records from the Croissant dataset schema using only `@id` references to RecordSets and fields.
- We performed a data overview, extracted tabular data into DataFrames, and demonstrated basic filtering, normalization, and aggregation.
- Data visualization highlighted numeric field distributions and group comparisons.

Further analysis can target domain-specific properties and deeper statistical/proteomic questions using this framework.